# الفصل 37: نماذج الرؤية واللغة: VLMs

> كل ما تعلمته عن CLIP وآليات الدمج في الفصلين السابقين كان الأساس النظري.

- **الجزء:** 8 — النماذج متعددة الوسائط
- **الفصل:** 37
- **المستوى:** متقدم
- **زمن القراءة:** 20 دقيقة
- **المتطلبات:** الفصل 36: مُشفِّرات الرؤية: ViT وCLIP
- **الوسوم:** LLaVA، BLIP-2، Flamingo، Visual Instruction Tuning

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-08-multimodal/ch37-vision-language-models.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-08-multimodal%2Fch37-vision-language-models.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-08-multimodal/ch37-vision-language-models.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-08-multimodal/ch37-vision-language-models.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-08-multimodal/ch37-vision-language-models.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-08-multimodal/ch37-vision-language-models.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** peft, torch, transformers
- **GPU:** يستخدم هذا الدفتر PyTorch أو نماذج كبيرة؛ يُفضّل GPU. يمكنك البدء بنماذج صغيرة على CPU، لكن التدريب الكامل سيكون بطيئًا.
- **بيانات/نماذج خارجية:** بعض الخلايا تتوقع ملفات أو نماذج محمّلة مسبقًا. عدّل المسارات أو حمّل البيانات قبل التشغيل.
- **خلايا ثقيلة:** قد تستغرق خلايا التدريب/التحميل وقتًا طويلًا. شغّلها خلية بخية، وقلّل `batch_size` أو استخدم نموذجًا أصغر عند الحاجة.
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [
    ("peft", "peft"),
    ("torch", "torch"),
    ("transformers", "transformers")
]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-08-multimodal/ch37-vision-language-models.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-08-multimodal%2Fch37-vision-language-models.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-08-multimodal/ch37-vision-language-models.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-08-multimodal/ch37-vision-language-models.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-08-multimodal/ch37-vision-language-models.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-08-multimodal/ch37-vision-language-models.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> كل ما تعلمته عن CLIP وآليات الدمج في الفصلين السابقين كان الأساس النظري.

> **بيانات الفصل**
>
> - **الجزء:** 8 — النماذج متعددة الوسائط
> - **المستوى:** متقدم
> - **زمن القراءة:** 20 دقيقة
> - **المتطلبات:** الفصل 36: مُشفِّرات الرؤية: ViT وCLIP


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم عائلة LLaVA — أبسط وأهم تصميم مرجعي
- فهم Q-Former وPerceiver Resampler: تقليل عدد الرموز البصرية
- فهم LLaVA


## 1. لماذا يهمّك هذا الموضوع؟

كل ما تعلمته عن CLIP وآليات الدمج في الفصلين السابقين كان الأساس النظري. هذا الفصل هو حيث **تُبنى الأنظمة الحقيقية**: كيف تحوّل نموذج لغوي نصّي بحت (مثل LLaMA أو Vicuna) إلى مساعد يرى الصور ويفهمها ويتحدث عنها؟ هذا بالضبط ما فعلته فرق LLaVA وFlamingo وQwen-VL وGPT-4V، وهي البنية الأساسية وراء كل "مساعد يفهم الصور" تستخدمه اليوم.


## 2. الخريطة الذهنية


```text
نموذج VLM حديث =
    [مُرمِّز رؤية مجمَّد غالبًا (Vision Encoder)]
            │ (patch embeddings)
            ▼
    [طبقة إسقاط/ربط (Connector/Projector)]
            │ (visual tokens بنفس بُعد embedding النص)
            ▼
    [نموذج لغوي كبير (LLM) — يُدرَّب أو يُضبط]
            │
            ▼
    [إجابة نصية مشروطة بالصورة + النص]
```


🟢 **طبقة الحدس**: فكّر في الصورة كأنها "فقرة نصية مُشفَّرة" تُضاف إلى بداية المحادثة — النموذج اللغوي لا "يرى" حرفيًا، بل يتعامل مع تمثيلات رقمية (visual tokens) تم إسقاطها لتبدو كأنها embeddings نصية عادية.


## 3. الشرح المتدرج

### 🟢 أ) عائلة LLaVA — أبسط وأهم تصميم مرجعي

**الفكرة المحورية**: بدلاً من تدريب نموذج رؤية-لغة ضخم من الصفر، نأخذ:
1. مُرمِّز رؤية مجمَّد (CLIP ViT-L/14)
2. طبقة إسقاط بسيطة (MLP من طبقتين)
3. LLM جاهز (Vicuna/LLaMA)

ونُدرّب **فقط** طبقة الإسقاط (وأحيانًا نضبط LLM لاحقًا بـ LoRA).

**مراحل تدريب LLaVA (مهمة جدًا لفهم منهجية VLM الحديثة بأكملها)**:

| المرحلة | الهدف | ما يُدرَّب | البيانات |
|---|---|---|---|
| **1. Feature Alignment (Pretraining)** | تعليم المُسقِط تحويل visual features لمساحة اللغة | طبقة الإسقاط فقط (LLM و Vision Encoder مجمّدان) | أزواج صورة-تعليق بسيطة (CC3M مُنقّاة ~595K) |
| **2. Visual Instruction Tuning** | تعليم النموذج اتّباع التعليمات مع الصور | طبقة الإسقاط + LLM (كامل أو LoRA) | بيانات تعليمات مُولَّدة بـ GPT-4 (LLaVA-Instruct-150K) |

> 💡 **لماذا مرحلتان؟** إذا دربت كل شيء معًا من البداية، الإشارة العشوائية من طبقة الإسقاط غير المُدرَّبة ستُفسد أوزان LLM المُدرَّبة مسبقًا (Catastrophic Forgetting المبكر).

### 🟡 ب) الكود الكامل: بناء نموذج LLaVA مبسّط من الصفر


In [ ]:
import torch
import torch.nn as nn
from transformers import CLIPVisionModel, AutoModelForCausalLM, AutoTokenizer

class VisionLanguageConnector(nn.Module):
    """
    طبقة الإسقاط (Projector) — القلب النابض لعائلة LLaVA.
    تحوّل visual features (من CLIP) إلى نفس فضاء embeddings النص.
    """
    def __init__(self, vision_hidden_size: int, llm_hidden_size: int,
                 projector_type: str = "mlp2x_gelu"):
        super().__init__()
        if projector_type == "linear":
            # أبسط تصميم (LLaVA-1.0)
            self.proj = nn.Linear(vision_hidden_size, llm_hidden_size)
        elif projector_type == "mlp2x_gelu":
            # تصميم LLaVA-1.5 — تحسّن ملحوظ بتكلفة زهيدة
            self.proj = nn.Sequential(
                nn.Linear(vision_hidden_size, llm_hidden_size),
                nn.GELU(),
                nn.Linear(llm_hidden_size, llm_hidden_size)
            )
        else:
            raise ValueError(f"نوع غير مدعوم: {projector_type}")

    def forward(self, visual_features: torch.Tensor) -> torch.Tensor:
        # visual_features: [batch, num_patches, vision_hidden_size]
        return self.proj(visual_features)  # -> [batch, num_patches, llm_hidden_size]

class SimpleLLaVA(nn.Module):
    """
    تطبيق تعليمي مبسّط لمعمارية LLaVA الكاملة.
    """
    def __init__(self, vision_model_name="openai/clip-vit-large-patch14-336",
                 llm_model_name="lmsys/vicuna-7b-v1.5",
                 freeze_vision=True, use_lora_llm=True):
        super().__init__()

        # 1) مُرمِّز الرؤية — يبقى مجمّدًا في أغلب التصاميم
        self.vision_encoder = CLIPVisionModel.from_pretrained(vision_model_name)
        if freeze_vision:
            for p in self.vision_encoder.parameters():
                p.requires_grad = False

        # 2) طبقة الربط
        vision_hidden = self.vision_encoder.config.hidden_size   # 1024 لـ ViT-L

        # 3) النموذج اللغوي
        self.llm = AutoModelForCausalLM.from_pretrained(
            llm_model_name, torch_dtype=torch.bfloat16
        )
        self.tokenizer = AutoTokenizer.from_pretrained(llm_model_name)
        llm_hidden = self.llm.config.hidden_size

        self.connector = VisionLanguageConnector(vision_hidden, llm_hidden)

        # رمز خاص يمثّل مكان إدراج الصورة في النص
        self.tokenizer.add_special_tokens({"additional_special_tokens": ["<image>"]})
        self.llm.resize_token_embeddings(len(self.tokenizer))
        self.image_token_id = self.tokenizer.convert_tokens_to_ids("<image>")

        if use_lora_llm:
            from peft import LoraConfig, get_peft_model
            lora_cfg = LoraConfig(
                r=128, lora_alpha=256,
                target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
                lora_dropout=0.05, task_type="CAUSAL_LM"
            )
            self.llm = get_peft_model(self.llm, lora_cfg)

    def encode_image(self, pixel_values: torch.Tensor) -> torch.Tensor:
        """استخراج ملامح الصورة وإسقاطها إلى فضاء اللغة"""
        with torch.no_grad() if not self.vision_encoder.training else torch.enable_grad():
            vision_outputs = self.vision_encoder(pixel_values)
        # نتجاهل CLS token، نأخذ patch tokens فقط (مثل LLaVA)
        patch_features = vision_outputs.last_hidden_state[:, 1:, :]
        return self.connector(patch_features)   # [batch, 576, llm_hidden] لصورة 336x336

    def merge_text_and_image_embeds(self, input_ids, pixel_values, attention_mask):
        """
        الخطوة الأهم: استبدال موضع <image> في النص بـ 576 visual token
        """
        text_embeds = self.llm.get_input_embeddings()(input_ids)
        image_embeds = self.encode_image(pixel_values)  # [B, 576, H]

        batch_size = input_ids.shape[0]
        merged_embeds_list = []
        merged_mask_list = []

        for b in range(batch_size):
            img_pos = (input_ids[b] == self.image_token_id).nonzero(as_tuple=True)[0]
            if len(img_pos) == 0:
                merged_embeds_list.append(text_embeds[b])
                merged_mask_list.append(attention_mask[b])
                continue
            pos = img_pos[0].item()
            merged = torch.cat([
                text_embeds[b, :pos],
                image_embeds[b],
                text_embeds[b, pos+1:]
            ], dim=0)
            new_mask = torch.cat([
                attention_mask[b, :pos],
                torch.ones(image_embeds.shape[1], device=attention_mask.device),
                attention_mask[b, pos+1:]
            ], dim=0)
            merged_embeds_list.append(merged)
            merged_mask_list.append(new_mask)

        # padding للدفعة لتوحيد الطول
        max_len = max(e.shape[0] for e in merged_embeds_list)
        padded_embeds = torch.stack([
            torch.nn.functional.pad(e, (0, 0, 0, max_len - e.shape[0]))
            for e in merged_embeds_list
        ])
        padded_mask = torch.stack([
            torch.nn.functional.pad(m, (0, max_len - m.shape[0]))
            for m in merged_mask_list
        ])
        return padded_embeds, padded_mask

    def forward(self, input_ids, pixel_values, attention_mask, labels=None):
        inputs_embeds, merged_mask = self.merge_text_and_image_embeds(
            input_ids, pixel_values, attention_mask
        )
        outputs = self.llm(
            inputs_embeds=inputs_embeds,
            attention_mask=merged_mask,
            labels=labels   # يُحسب فقط على أجزاء النص، visual tokens تُقنَّع بـ -100
        )
        return outputs


> ⚠️ **خطأ شائع جدًا**: نسيان قناع (mask) الـ `labels` الموافقة لـ visual tokens بقيمة `-100` بحيث لا تُحتسب في دالة الخسارة. الصورة **مدخل شرطي فقط**، وليست هدفًا للتنبؤ.

### 🔴 ج) طبقة التعمّق: Flamingo وآلية Gated Cross-Attention

على عكس LLaVA (الذي يحقن visual tokens مباشرة في تسلسل الإدخال)، يستخدم **Flamingo** (DeepMind) تصميمًا مختلفًا جوهريًا: **طبقات Cross-Attention مُدخَلة بين طبقات الـ LLM المجمّدة**.


In [ ]:
class GatedCrossAttentionBlock(nn.Module):
    """
    نمط Flamingo: يُدخَل بين طبقات LLM المجمَّدة بالكامل
    بوابة tanh تبدأ من الصفر لضمان عدم إفساد النموذج المُدرَّب مسبقًا في البداية
    """
    def __init__(self, dim, num_heads=8):
        super().__init__()
        self.cross_attn = nn.MultiheadAttention(dim, num_heads, batch_first=True)
        self.attn_gate = nn.Parameter(torch.zeros(1))   # tanh-gating — البداية = صفر تأثير!

        self.ff = nn.Sequential(
            nn.Linear(dim, dim * 4), nn.GELU(), nn.Linear(dim * 4, dim)
        )
        self.ff_gate = nn.Parameter(torch.zeros(1))
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)

    def forward(self, text_hidden, visual_features):
        attn_out, _ = self.cross_attn(
            query=self.norm1(text_hidden), key=visual_features, value=visual_features
        )
        text_hidden = text_hidden + torch.tanh(self.attn_gate) * attn_out
        ff_out = self.ff(self.norm2(text_hidden))
        text_hidden = text_hidden + torch.tanh(self.ff_gate) * ff_out
        return text_hidden


> 💡 **لماذا `tanh(gate)` مبدئيًا = صفر؟** هذا يضمن أن النموذج في بداية التدريب يتصرف **تمامًا** كأن طبقات Cross-Attention غير موجودة — استقرار تدريبي أعلى بكثير من حقن مباشر.

### 🔴 د) Q-Former وPerceiver Resampler: تقليل عدد الرموز البصرية

حقن كل رقع الصورة مباشرة في LLM بسيط وفعّال، لكنه مكلف عندما ترتفع دقة الصورة أو
عندما نمرّر عدة صور في المحادثة نفسها. الحل البديل هو **ضغط الرموز البصرية** إلى
عدد ثابت صغير قبل إدخالها إلى LLM:

- **Q-Former (BLIP-2):** مجموعة صغيرة من الاستعلامات القابلة للتعلّم (مثلًا 32
  استعلامًا) تنتبه إلى تمثيلات الرقع عبر Cross-Attention، فتستخرج «ملخصًا» ثابت
  الطول بغض النظر عن دقة الصورة.
- **Perceiver Resampler (Flamingo):** الفكرة نفسها بصياغة مختلفة: استعلامات ثابتة
  تُجمِّع معلومات من الرقع، ثم تُمرَّر إلى طبقات Cross-Attention المدمجة في LLM.

المقايضة واضحة: عدد رموز أقل يعني سياقًا أرخص وأسرع، لكنه قد يفقد تفاصيل مكانية
دقيقة (نص صغير في مستند، إحداثيات، أرقام في جدول). لذلك تختار التصاميم الحديثة
بين الحقن المباشر (LLaVA) والضغط (BLIP-2/Flamingo) حسب المهمة.


## 4. مثال محلول خطوة بخطوة

**المثال:** نريد ربط مُشفِّر رؤية `CLIP ViT-L/14-336` بنموذج لغوي بـ 7B
معاملات (بُعد مخفي 4096). لنحسب تكلفة الربط قبل كتابة أي كود.

**الخطوة 1 — عدد الرقع البصرية:**

$$
N_{\text{patches}} = \left(\frac{336}{14}\right)^2 = 24^2 = 576
$$

أي أن الصورة الواحدة تتحول إلى 576 رمزًا بصريًا (من دون [CLS]).

**الخطوة 2 — معاملات طبقة الربط (MLP من طبقتين):**

- الطبقة الأولى: $1024 \times 4096 + 4096 \approx 4.20$ مليون معامل.
- الطبقة الثانية: $4096 \times 4096 + 4096 \approx 16.78$ مليون معامل.
- الإجمالي: حوالي **21 مليون معامل**، أي أقل من 0.3% من حجم LLM.

**الخطوة 3 — أثر الرموز البصرية على السياق:**

إذا كان نص السؤال 128 رمزًا، يصبح طول التسلسل:

$$
128 + 576 = 704 \text{ tokens}
$$

وكلفة الانتباه تتناسب تقريبًا مع مربع الطول:

$$
\left(\frac{704}{128}\right)^2 \approx 30\times
$$

أي أن إضافة صورة واحدة ترفع كلفة الانتباه نحو **30 ضعفًا** مقارنة بالسؤال النصي
وحده. هذا هو السبب الاقتصادي وراء ابتكار Q-Former وPerceiver Resampler.

**الخطوة 4 — أثر KV Cache عند الاستدلال:**

لنموذج بـ 32 طبقة وبُعد مخفي 4096 ودقة bf16:

$$
2 \times 32 \times 4096 \times 2 \approx 512 \text{ KB/token}
$$

أي أن 576 رمزًا بصريًا تضيف نحو **288 MB** إلى ذاكرة KV Cache لكل تسلسل. مع
عدة مستخدمين متزامنين يصبح ضغط الذاكرة كبيرًا، لذا فإن تقليل الرموز البصرية
ليس تحسينًا تجميليًا بل قرار تكلفة أساسي.

**الخلاصة:** ابدأ بالحقن المباشر إن كانت الدقة متوسطة والمهمة تحتاج تفاصيل،
وانتقل إلى الضغط عندما تتعدد الصور أو ترتفع الدقة أو تضيق ميزانية الذاكرة.


## 5. نصائح وحيل

- **ابدأ بتجميد مُشفِّر الرؤية.** درّب طبقة الربط فقط في المرحلة الأولى؛ هذا يقلل الذاكرة والزمن ويمنع تدمير التمثيلات البصرية العامة.
- **اتبع التدريب على مرحلتين.** محاذاة (Captioning) ثم تعليمات بصرية (Instruction Tuning)؛ خلطهما من البداية يربك طبقة الربط غير المدربة.
- **اقنع رموز الصورة في دالة الخسارة.** استبدل labels المواضع البصرية بـ -100 حتى لا يتعلم النموذج «توليد الصورة» بدل الإجابة عن السؤال.
- **اضبط عدد الرموز البصرية حسب المهمة.** 576 رمزًا مناسب لصورة واحدة عالية الدقة، لكنه مكلف مع 4–8 صور؛ استخدم Resampler أو دقة أقل عند الحاجة.
- **استخدم LoRA على LLM** بدل فتح كل المعاملات عندما تكون الموارد محدودة.
- **وحّد preprocessing** بين تدريب المُشفِّر والاستدلال (resize، crop، mean/std).
- **نوّع بيانات التعليمات:** أسئلة قصيرة، وصف تفصيلي، مقارنة، عدّ، قراءة نص، واستدلال مكاني؛ الاكتفاء بنوع واحد ينتج نموذجًا ضيقًا.
- **قِس الهلوسة البصرية مبكرًا** (POPE) لا الدقة العامة فقط؛ النموذج قد يجيب بطلاقة عن أشياء غير موجودة في الصورة.


## 6. أخطاء شائعة ومصائد

- **نسيان قناع labels للرموز البصرية:** يجعل النموذج يتدرب على إعادة إنتاج تمثيلات الصورة، فتتدهور جودة الإجابة.
- **تدريب مُشفِّر الرؤية من الصفر:** يحتاج بيانات ضخمة؛ استخدم CLIP/SigLIP مُدرَّبًا مسبقًا وابدأ من هناك.
- **استخدام [CLS] فقط:** يفقد التفاصيل المكانية اللازمة لقراءة الجداول والمستندات.
- **عدد رموز بصرية غير واقعي:** 576 رمزًا × عدة صور قد يتجاوز سياق LLM أو يضاعف زمن الاستدلال دون مكسب حقيقي.
- **عدم تطابق المعالجة المسبقة:** صورة مقصوصة أو مطبَّعة بشكل مختلف عن تدريب المُشفِّر تُنتج تمثيلات رديئة حتى لو كان الكود صحيحًا.
- **تجاهل نسبة الأبعاد:** قصّ aggressif يحذف أجزاء مهمة من الصورة (نص، وجوه).
- **استخدام LLM أساسي غير مُدرَّب على التعليمات:** المرحلة الثانية تحتاج نموذجًا يتبع التعليمات أصلًا، وإلا تتحول المشكلة إلى مشكلة SFT عامة.
- **تقييم على بيانات التدريب فقط:** يقيس الحفظ لا القدرة؛ افصل مجموعة تقييم بصرية مستقلة تمامًا.


## 7. جدول مقارنة

| النموذج | آلية الدمج | Vision Encoder | هل LLM مجمّد؟ | نقطة القوة |
|---|---|---|---|---|
| **LLaVA-1.5/1.6** | حقن مباشر لـ visual tokens | CLIP ViT-L/14-336 | لا (LoRA/Full FT) | بساطة، كفاءة تدريب عالية |
| **Flamingo** | Gated Cross-Attention | NFNet/CLIP | نعم (مجمّد بالكامل) | يدعم تعدد صور بسهولة (Few-shot) |
| **BLIP-2 / Q-Former** | Querying Transformer (32 query مُتعلَّمة) | ViT-g | نعم | كفاءة بارامترات عالية جدًا |
| **Qwen-VL** | Cross-Attention + تجزئة دقيقة للصور | ViT مخصص | لا | دعم المواقع المكانية (bounding boxes) |
| **GPT-4V / GPT-4o** | غير معلن (يُرجَّح: tokenization موحّد) | غير معلن | - | أداء عام الأفضل |
| **PaLI-X** | Encoder-Decoder موحّد | ViT ضخم (22B) | لا | تدريب end-to-end كامل |

### 📊 مقاييس تقييم VLM الحديثة

| المقياس | ماذا يقيس |
|---|---|
| **MMMU** | الاستدلال متعدد التخصصات على مستوى جامعي |
| **MMBench** | قدرات إدراكية ومنطقية متنوعة (ثنائي اللغة) |
| **ChartQA / DocVQA** | فهم الرسوم البيانية والمستندات |
| **POPE** | قياس الهلوسة البصرية (Object Hallucination) |
| **MME** | تقييم شامل 14 مهمة فرعية |

---


## 8. ملخص أفضل الممارسات

- [ ] مُشفِّر الرؤية مُدرَّب مسبقًا ومجمَّد في المرحلة الأولى.
- [ ] طبقة الربط قابلة للتدريب، وحجمها موثَّق ومبرَّر.
- [ ] رموز الصورة مقنَّعة بـ -100 في دالة الخسارة.
- [ ] المعالجة المسبقة للصور مطابقة لتدريب المُشفِّر.
- [ ] عدد الرموز البصرية محسوب ومُختبر مع أطول سياق متوقع.
- [ ] مرحلتا التدريب (محاذاة ثم تعليمات) منفصلتان وموثَّقتان.
- [ ] تقييم على MMMU/MMBench/POPE أو ما يناسب مجال التطبيق.
- [ ] اختبار حالات «صورة بلا إجابة» و«سؤال عن شيء غير موجود» لقياس الهلوسة.
- [ ] قياس زمن الاستدلال والذاكرة مع صورة واحدة وعدة صور.
- [ ] خطة تراجع واضحة إذا تدهورت قدرات LLM النصية بعد الضبط.


## 9. أسئلة مقابلات

**س: لماذا نجمّد مُشفِّر الرؤية في معظم تصاميم VLM؟**

ج: لأن CLIP مُدرَّب على مليارات أزواج (صورة، نص) ويملك تمثيلًا بصريًا عامًا قويًا.
إعادة تدريبه على بيانات تعليمات أصغر بكثير قد تُدمِّر هذا التمثيل (نسيان كارثي)
مقابل مكسب محدود، كما أن backprop عبر ViT كبير مكلف حسابيًا.

**س: ما الفرق الجوهري بين LLaVA وFlamingo وBLIP-2؟**

ج: LLaVA يحقن رموز الصورة مباشرة في تسلسل LLM بعد إسقاط بسيط. Flamingo يضيف
طبقات Cross-Attention بوابة بين طبقات LLM المجمَّدة، ويحتفظ بعدد كبير من الرموز
البصرية. BLIP-2 يستخدم Q-Former لضغط الرقع إلى عدد ثابت صغير من الاستعلامات قبل
إدخالها إلى LLM. الفرق الأساسي: أين تُدمج الرؤية، وكم رمزًا تستهلك.

**س: متى تختار Q-Former بدل MLP بسيط؟**

ج: عندما تكون الصور عالية الدقة أو متعددة، أو عندما تضيق ميزانية الذاكرة/الزمن.
Q-Former يقلل الرموز إلى 32–64 مثلًا، لكنه يضيف تعقيدًا تدريبيًا وقد يفقد تفاصيل
مكانية دقيقة. للمهام التي تحتاج قراءة نص صغير أو إحداثيات، قد يكون الحقن المباشر
أفضل.

**س: كيف تعرف أن النموذج يعاني من الهلوسة البصرية؟**

ج: استخدم POPE أو مجموعة أسئلة «هل يوجد X في الصورة؟» مع صور تحتوي أو لا تحتوي
على X، وقارن الإجابات بالحقيقة. راقب أيضًا الإجابات الواثقة عن أشياء غير موجودة،
وأضف تدريبًا سلبيًا (أسئلة عن غياب العنصر) إن كانت الهلوسة مرتفعة.


## 10. تمرين عملي

**الهدف:** استبدال طبقة الربط البسيطة بـ Perceiver Resampler وتجربة أثره على
طول السياق والذاكرة.

1. نفّذ `PerceiverResampler` بعدد استعلامات ثابت (مثلًا 64) فوق تمثيلات الرقع.
2. اربطه بـ LLM صغير (TinyLlama أو Qwen2-0.5B) بدل نموذج 7B لتسريع التجربة.
3. قارن ثلاث تهيئات: Linear وMLP وResampler-64 على مجموعة VQA صغيرة.
4. سجّل: عدد الرموز البصرية، زمن الخطوة، ذاكرة GPU، ودقة الإجابات.
5. أجب: هل وُفِّرت الموارد؟ وهل انخفضت الدقة؟ وفي أي نوع من الأسئلة؟
6. **(تحدٍّ إضافي):** جرّب Resampler بعدد استعلامات 16 و32 و128 وارسم منحنى
   «الدقة مقابل عدد الرموز».


## 11. مراجع للتعمّق

- Liu et al. (2023): *Visual Instruction Tuning* (LLaVA).
- Li et al. (2023): *BLIP-2: Bootstrapping Language-Image Pre-training with Frozen Image Encoders and Large Language Models*.
- Alayrac et al. (2022): *Flamingo: a Visual Language Model for Few-Shot Learning*.
- Bai et al. (2023): *Qwen-VL: A Versatile Vision-Language Model for Understanding, Localization, Text Reading, and Beyond*.
- Zhai et al. (2023): *Sigmoid Loss for Language Image Pre-Training* (SigLIP).
- Fu et al. (2023): *MMMU: A Massive Multi-discipline Multimodal Understanding and Reasoning Benchmark*.
- Liu et al. (2023): *MMBench: Is Your Multi-modal Model an All-around Player?*
- Li et al. (2023): *Evaluating Object Hallucination in Large Vision-Language Models* (POPE).
- Hugging Face TRL — https://huggingface.co/docs/trl


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — النماذج متعددة الوسائط / الفصل 37: نماذج الرؤية واللغة: VLMs.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
